## Gold tableの作成

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.ml.functions import vector_to_array

import os
import mlflow
import mlflow.spark

In [0]:
CUSTOMER_TABLE = "workspace.bank.silver_customer"
TRANSACTION_TABLE = "workspace.bank.silver_transaction_monthly"
CRM_TABLE = "workspace.bank.silver_crm_activity"
GOLD_TABLE = "workspace.bank.gold_company_support_features"

# 登録済みモデルのVersion番号に合わせて変更
MODEL_URI = "models:/workspace.bank.next_month_overdue_logistic_regression/1"

MLFLOW_TMP_DIR = "/Volumes/workspace/bank/vol/mlflow_tmp"
os.environ["MLFLOW_DFS_TMP"] = MLFLOW_TMP_DIR
dbutils.fs.mkdirs(MLFLOW_TMP_DIR)
mlflow.set_registry_uri("databricks-uc")

In [0]:
customer_df = spark.table(CUSTOMER_TABLE)
transaction_df = spark.table(TRANSACTION_TABLE)
crm_df = spark.table(CRM_TABLE)

print("customer:", customer_df.count())
print("transaction:", transaction_df.count())
print("crm:", crm_df.count())

In [0]:
latest_month_window = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("transaction_month").desc())
)

oldest_month_window = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("transaction_month").asc())
)

transaction_ranked_df = (
    transaction_df
    .withColumn("latest_month_rank", F.row_number().over(latest_month_window))
    .withColumn("oldest_month_rank", F.row_number().over(oldest_month_window))
)

In [0]:

# モデルが学習時に受け取った元の列名を保持
latest_model_input_df = (
    transaction_ranked_df
    .filter(F.col("latest_month_rank") == 1)
    .select(
        "customer_id",
        "transaction_month",
        "monthly_inflow",
        "monthly_outflow",
        "deposit_balance",
        "loan_balance",
        "overdue_days",
    )
)

latest_transaction_df = (
    latest_model_input_df
    .select(
        "customer_id",
        F.col("transaction_month").alias("latest_transaction_month"),
        F.col("monthly_inflow").alias("latest_monthly_inflow"),
        F.col("monthly_outflow").alias("latest_monthly_outflow"),
        F.col("deposit_balance").alias("latest_deposit_balance"),
        F.col("loan_balance").alias("latest_loan_balance"),
        F.col("overdue_days").alias("latest_overdue_days"),
    )
)

oldest_transaction_df = (
    transaction_ranked_df
    .filter(F.col("oldest_month_rank") == 1)
    .select(
        "customer_id",
        F.col("transaction_month").alias("oldest_transaction_month"),
        F.col("monthly_inflow").alias("oldest_monthly_inflow"),
        F.col("deposit_balance").alias("oldest_deposit_balance"),
    )
)


In [0]:

registered_model = mlflow.spark.load_model(
    model_uri=MODEL_URI,
    dfs_tmpdir=MLFLOW_TMP_DIR,
)

prediction_raw_df = registered_model.transform(latest_model_input_df)

model_prediction_df = (
    prediction_raw_df
    .withColumn(
        "next_month_overdue_probability",
        vector_to_array(F.col("probability"))[1],
    )
    .withColumn("model_prediction", F.col("prediction").cast("int"))
    .withColumn("prediction_generated_at", F.current_timestamp())
    .select(
        "customer_id",
        "next_month_overdue_probability",
        "model_prediction",
        "prediction_generated_at",
    )
)

In [0]:
transaction_feature_df = (
    latest_transaction_df
    .join(oldest_transaction_df, on="customer_id", how="left")
    .join(model_prediction_df, on="customer_id", how="left")
    .withColumn(
        "inflow_change_rate",
        F.when(
            F.col("oldest_monthly_inflow") > 0,
            (F.col("latest_monthly_inflow") - F.col("oldest_monthly_inflow"))
            / F.col("oldest_monthly_inflow"),
        ),
    )
    .withColumn(
        "deposit_change_rate",
        F.when(
            F.col("oldest_deposit_balance") > 0,
            (F.col("latest_deposit_balance") - F.col("oldest_deposit_balance"))
            / F.col("oldest_deposit_balance"),
        ),
    )
    .withColumn(
        "net_cash_flow",
        F.col("latest_monthly_inflow") - F.col("latest_monthly_outflow"),
    )
    .withColumn(
        "loan_to_deposit_ratio",
        F.when(
            F.col("latest_deposit_balance") > 0,
            F.col("latest_loan_balance") / F.col("latest_deposit_balance"),
        ),
    )
)

In [0]:
latest_crm_window = (
    Window.partitionBy("customer_id")
    .orderBy(F.col("activity_date").desc())
)

latest_crm_df = (
    crm_df
    .withColumn("crm_rank", F.row_number().over(latest_crm_window))
    .filter(F.col("crm_rank") == 1)
    .select(
        "customer_id",
        F.col("activity_date").alias("last_activity_date"),
        F.col("meeting_note").alias("latest_meeting_note"),
        F.col("customer_concern").alias("latest_customer_concern"),
        F.col("next_action").alias("latest_next_action"),
    )
)

EVALUATION_DATE = "2026-07-20"

crm_feature_df = (
    latest_crm_df
    .withColumn(
        "days_since_last_activity",
        F.datediff(
            F.to_date(F.lit(EVALUATION_DATE)),
            F.col("last_activity_date"),
        ),
    )
)

In [0]:
gold_base_df = (
    customer_df
    .join(transaction_feature_df, on="customer_id", how="left")
    .join(crm_feature_df, on="customer_id", how="left")
)

In [0]:
gold_explanation_df = (
    gold_base_df
    .withColumn(
        "inflow_risk_score",
        F.when(F.col("inflow_change_rate") <= -0.30, 30)
        .when(F.col("inflow_change_rate") <= -0.15, 20)
        .when(F.col("inflow_change_rate") <= -0.05, 10)
        .otherwise(0),
    )
    .withColumn(
        "deposit_risk_score",
        F.when(F.col("deposit_change_rate") <= -0.20, 25)
        .when(F.col("deposit_change_rate") <= -0.10, 15)
        .when(F.col("deposit_change_rate") <= -0.05, 8)
        .otherwise(0),
    )
    .withColumn(
        "overdue_risk_score",
        F.when(F.col("latest_overdue_days") >= 10, 25)
        .when(F.col("latest_overdue_days") >= 5, 15)
        .when(F.col("latest_overdue_days") >= 1, 8)
        .otherwise(0),
    )
    .withColumn(
        "activity_risk_score",
        F.when(F.col("days_since_last_activity") >= 90, 20)
        .when(F.col("days_since_last_activity") >= 60, 12)
        .when(F.col("days_since_last_activity") >= 30, 5)
        .otherwise(0),
    )
    # モデル確率を0〜100点に変換
    .withColumn(
        "support_need_score",
        F.round(
            F.coalesce(F.col("next_month_overdue_probability"), F.lit(0.0)) * 100,
            1,
        ),
    )
    .withColumn(
        "support_priority",
        F.when(F.col("next_month_overdue_probability") >= 0.70, "High")
        .when(F.col("next_month_overdue_probability") >= 0.40, "Medium")
        .otherwise("Low"),
    )
)

In [0]:
gold_reason_df = (
    gold_explanation_df
    .withColumn(
        "support_reasons",
        F.concat_ws(
            " / ",
            F.when(
                F.col("inflow_change_rate") <= -0.15,
                F.concat(
                    F.lit("入金額が"),
                    F.format_number(F.abs(F.col("inflow_change_rate") * 100), 1),
                    F.lit("%減少"),
                ),
            ),
            F.when(
                F.col("deposit_change_rate") <= -0.10,
                F.concat(
                    F.lit("預金残高が"),
                    F.format_number(F.abs(F.col("deposit_change_rate") * 100), 1),
                    F.lit("%減少"),
                ),
            ),
            F.when(
                F.col("latest_overdue_days") > 0,
                F.concat(
                    F.lit("延滞"),
                    F.col("latest_overdue_days").cast("string"),
                    F.lit("日"),
                ),
            ),
            F.when(
                F.col("days_since_last_activity") >= 30,
                F.concat(
                    F.lit("最終営業接点から"),
                    F.col("days_since_last_activity").cast("string"),
                    F.lit("日経過"),
                ),
            ),
        ),
    )
)

In [0]:

gold_company_support_features_df = (
    gold_reason_df
    .select(
        "customer_id",
        "company_name",
        "industry",
        "annual_sales",
        "employee_count",
        "branch_name",
        "relationship_manager",
        "latest_transaction_month",
        "latest_monthly_inflow",
        "latest_monthly_outflow",
        "net_cash_flow",
        "latest_deposit_balance",
        "latest_loan_balance",
        "loan_to_deposit_ratio",
        "latest_overdue_days",
        "inflow_change_rate",
        "deposit_change_rate",
        "last_activity_date",
        "days_since_last_activity",
        "latest_meeting_note",
        "latest_customer_concern",
        "latest_next_action",
        "next_month_overdue_probability",
        "model_prediction",
        "prediction_generated_at",
        "inflow_risk_score",
        "deposit_risk_score",
        "overdue_risk_score",
        "activity_risk_score",
        "support_need_score",
        "support_priority",
        "support_reasons",
    )
)


In [0]:
display(
    gold_company_support_features_df
    .orderBy(F.col("support_need_score").desc())
)

(
    gold_company_support_features_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(GOLD_TABLE)
)

print(f"保存完了: {GOLD_TABLE}")